# 06. Codeforces analysis + cross-platform comparison (RQ2)

Codeforces users' PFP type by rating distribution and cross-platform comparison with GitHub results.

**RQ2**: Anime PFP user's rating significantly higher than other types?

**Input**: `data/processed/codeforces_classified.csv`, `data/processed/classified_3cat.csv`

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

PROJECT_DIR = Path('../').resolve()
DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROC_DIR = DATA_DIR / 'processed'
FIG_DIR = PROJECT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

cf = pd.read_csv(PROC_DIR / 'codeforces_classified.csv')
print(f'Codeforces user: {len(cf)} users')
print(f'PFP type:\n{cf["profile_type"].value_counts()}')

## 1. PFP type by rating distribution

In [ ]:
colors = {'Anime': '#ff6b6b', 'Default': '#c0c0c0', 'Photo': '#4ecdc4'}
order = ['Anime', 'Photo', 'Default']

print('=== PFP type by rating descriptive statistics ===')
print(cf.groupby('profile_type')['rating'].describe()[['count', 'mean', '50%', 'std']].rename(columns={'50%': 'median'}).round(1))
print()
print(cf.groupby('profile_type')['maxRating'].describe()[['count', 'mean', '50%', 'std']].rename(columns={'50%': 'median'}).round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ['rating', 'maxRating'], ['Current Rating', 'Max Rating']):
    for ptype in order:
        data = cf[cf['profile_type'] == ptype][metric].dropna()
        ax.hist(data, bins=50, alpha=0.5, label=f'{ptype} (n={len(data)})',
                color=colors[ptype], density=True)
    ax.set_xlabel('Rating')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Codeforces PFP type by rating distribution', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cf_rating_distribution.png', dpi=150)
plt.show()

In [ ]:
# Box plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
palette = [colors[t] for t in order]

for ax, metric, title in zip(axes, ['rating', 'maxRating'], ['Current Rating', 'Max Rating']):
    sns.boxplot(data=cf, x='profile_type', y=metric, order=order, hue='profile_type', legend=False,
                palette=palette, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('')

plt.suptitle('Codeforces PFP type by rating comparison', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cf_rating_boxplot.png', dpi=150)
plt.show()

## 2. Rank bin by Anime PFP ratio

In [ ]:
# Codeforces rank order
rank_order = ['newbie', 'pupil', 'specialist', 'expert', 'candidate master',
              'master', 'international master', 'grandmaster', 
              'international grandmaster', 'legendary grandmaster']

# rank — rows with rank only
cf_ranked = cf[cf['rank'].isin(rank_order)].copy()
ct = pd.crosstab(cf_ranked['rank'], cf_ranked['profile_type'], normalize='index') * 100

# sort by rank order
ct = ct.reindex([r for r in rank_order if r in ct.index])

print('=== Rank by PFP type ratio (%) ===')
print(ct[order].round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Anime PFP ratio bar chart
if 'Anime' in ct.columns:
    anime_pct = ct['Anime']
    x = range(len(anime_pct))
    bars = ax.bar(x, anime_pct, color='#ff6b6b', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(anime_pct.index, rotation=30, ha='right')
    ax.set_ylabel('Anime PFP ratio (%)')
    ax.set_title('Codeforces Rank by Anime PFP ratio — "Anime PFP = pro" hypothesis test', fontsize=14)
    
    # show values on each bar
    for bar, val in zip(bars, anime_pct):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'cf_anime_by_rank.png', dpi=150)
plt.show()

## 3. statistical test

In [ ]:
anime_cf = cf[cf['profile_type'] == 'Anime']
default_cf = cf[cf['profile_type'] == 'Default']
normal_cf = cf[cf['profile_type'] == 'Photo']

# Kruskal-Wallis
for metric in ['rating', 'maxRating']:
    h, p = stats.kruskal(
        anime_cf[metric].dropna(),
        default_cf[metric].dropna(),
        normal_cf[metric].dropna()
    )
    n = len(cf[metric].dropna())
    eta_sq = max(0, (h - 2) / (n - 3))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'{metric}: H={h:.2f}, p={p:.2e}, η²={eta_sq:.4f} {sig}')

# Anime vs non-anime Mann-Whitney
print('\n=== Anime vs non-anime ===')
non_anime_cf = cf[cf['profile_type'] != 'Anime']
for metric in ['rating', 'maxRating']:
    u, p = stats.mannwhitneyu(anime_cf[metric].dropna(), non_anime_cf[metric].dropna(), alternative='two-sided')
    n1, n2 = len(anime_cf[metric].dropna()), len(non_anime_cf[metric].dropna())
    cliff_d = (2 * u) / (n1 * n2) - 1
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"{metric}: U={u:.0f}, p={p:.2e}, Cliff's δ={cliff_d:.3f} {sig}")
    print(f"  Anime median: {anime_cf[metric].median()}, non-anime median: {non_anime_cf[metric].median()}")

## 4. GitHub vs Codeforces cross-platform comparison

In [ ]:
gh = pd.read_csv(PROC_DIR / 'classified_3cat.csv')

# each platform's Anime PFP ratio
gh_anime_pct = (gh['profile_type'] == 'Anime').mean() * 100
cf_anime_pct = (cf['profile_type'] == 'Anime').mean() * 100

print('=== GitHub vs Codeforces cross-platform comparison ===')
print(f'\noverall Anime PFP ratio:')
print(f'  GitHub:     {gh_anime_pct:.1f}% ({(gh["profile_type"]=="Anime").sum()}/{len(gh)})')
print(f'  Codeforces: {cf_anime_pct:.1f}% ({(cf["profile_type"]=="Anime").sum()}/{len(cf)})')

# comparison table
comparison = pd.DataFrame({
    'Platform': ['GitHub', 'Codeforces'],
    'Total Users': [len(gh), len(cf)],
    'Anime %': [gh_anime_pct, cf_anime_pct],
    'Default %': [(gh['profile_type']=='Default').mean()*100, (cf['profile_type']=='Default').mean()*100],
    'Normal %': [(gh['profile_type']=='Photo').mean()*100, (cf['profile_type']=='Photo').mean()*100],
})
comparison.round(1)

In [ ]:
# side-by-side comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, data, title in zip(axes, [gh, cf], ['GitHub (activity metric)', 'Codeforces (rating)']):
    counts = data['profile_type'].value_counts()
    ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
           colors=[colors.get(t, '#999') for t in counts.index],
           startangle=90, textprops={'fontsize': 12})
    ax.set_title(f'{title}\n(n={len(data):,})', fontsize=13)

plt.suptitle('GitHub vs Codeforces PFP type distribution comparison', fontsize=15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cross_comparison.png', dpi=150)
plt.show()

## 5. Final Conclusion

In [ ]:
print('='*60)
print('Final conclusion summary')
print('='*60)
print()
print('RQ1 (GitHub): PFP type and open-source activity metric correlation')
print('  → 03, 04 see notebook results')
print()
print('RQ2 (Codeforces): Anime PFP and CP rating correlation')
print(f'  → Anime PFP ratio: {cf_anime_pct:.1f}%')
print(f'  → Anime PFP median rating: {anime_cf["rating"].median()}')
print(f'  → non-anime PFP median rating: {non_anime_cf["rating"].median()}')
print()
print('cross-platform comparison:')
print(f'  → GitHub Anime ratio: {gh_anime_pct:.1f}% / Codeforces Anime ratio: {cf_anime_pct:.1f}%')
print()
print('※ This study is correlation analysis only — no causation claims')
print('※ confounding variables(culture, experience, age, etc.) may have influence')